# Датасети: EDA

Той самий аналіз, що у вкладці «Датасети» сторінки: функція `build` зі `scripts/10_dataset_eda.py`.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from eaglewings.ds import report
pd.set_option('display.width', 200)
%matplotlib inline

In [2]:
import json
names = [n for g in json.load(open('../web/science/eda.json'))['groups'] for n in g['items']]
names

['burned_area_pixels',
 'matching_firms_effis',
 'fire_day_daily',
 'h3_risk_cells',
 'fire_spread_cells',
 'tp_fires',
 'tp_candidates',
 'logging_gr_pixels',
 'opera_gr_snapshots',
 'logging_gr_cells',
 'cv_benchmark_aerial',
 'cv_benchmark_ground',
 'cv_camera_zone',
 'cp_events',
 'xm_signals',
 'ar_fires',
 'as_areals']

## `burned_area_pixels`

In [3]:
o = report.dataset_eda('burned_area_pixels')
print(o['short'])
for s in o['seen']:
    print('·', s)
pd.DataFrame(o['columns'])[['n', 'doc', 'role', 'miss', 'uniq', 'min', 'med', 'max']]

493 181 пікселів лісу парку, 50 % із них втрачені у 2023–2024. Падіння NDVI добре відділяє згарище.
· Класи майже рівні: 50 % проти 50 %, тому F1 тут показовий.
· Частки в calib і test близькі: 50 % і 50 %. Спліт за блоками не перекосив лейбл.
· Найсильніша ознака на calib це падіння NDVI, сила 0,82. Лісистість майже нічого не дає (0,57).
· NDVI відсутній у 6,5 % пікселів, це пропуски MODIS.
· Сусідні пікселі майже однакові, тому 438 тисяч рядків це не 438 тисяч незалежних прикладів. Через це спліт і бутстреп ідуть за блоками.


,n,doc,role,miss,uniq,min,med,max
0,park,,інше,0.0000,3,NaN,NaN,NaN
1,row,рядок пікселя в сітці Hansen,місце,0.0000,1171,0.000,608.000,1171.000
2,col,колонка пікселя в сітці Hansen,місце,0.0000,2110,6.000,579.000,3499.000
3,h3,ідентифікатор гекса H3,ключ,0.0000,5356,NaN,NaN,NaN
4,block,просторовий блок для спліту,спліт,0.0000,280,NaN,NaN,NaN
5,split,частина спліту,спліт,0.0000,2,NaN,NaN,NaN
6,pix_ha,"площа пікселя, га",інше,0.0000,47,0.058,0.058,0.059
7,treecover,"лісистість 2000 року, %",ознака,0.0000,71,30.000,78.000,100.000
8,ndvi_pre,NDVI до пожежі,ознака,0.0648,3719,-0.179,0.741,0.986
9,ndvi_post,NDVI після пожежі,ознака,0.0647,3648,-0.146,0.379,0.863


## `fire_day_daily`

In [4]:
o = report.dataset_eda('fire_day_daily')
print(o['short'])
for s in o['seen']:
    print('·', s)
pd.DataFrame(o['columns'])[['n', 'doc', 'role', 'miss', 'uniq', 'min', 'med', 'max']]

П'ять років щоденної погоди, лише 327 дні з вогнем (6,0 %). Це рідкісна подія.
· Від 44 до 92 пожежних днів на рік: 2022 дає найбільше. На такому числі подій інтервали неминуче широкі.
· 19 % пожежних днів припадає на липень–вересень: сезон важить, але друга половина розкидана по інших місяцях.
· Найсильніші ознаки на 2021–2022: сухих днів поспіль (0,73), опади, мм (0,68). Вони описують сухість, а не вогонь напряму.
· Багато ознак дублюють одна одну: температура, дефіцит вологи й вологість ґрунту майже рухаються разом, кореляція за модулем від 0,84 до 0,96.
· Погода береться в одній точці, центрі парку, тому різницю всередині парку вона не показує.


,n,doc,role,miss,uniq,min,med,max
0,park,,інше,0.0,3,NaN,NaN,NaN
1,date,дата,час,0.0,1826,NaN,NaN,NaN
2,T2M_MAX,"максимальна температура, °C",ознака,0.0,2259,-1.040,20.570,42.830
3,T2M,"середня температура, °C",інше,0.0,2153,-3.180,15.360,34.260
4,RH2M,"вологість повітря, %",ознака,0.0,2721,28.400,70.465,98.070
5,WS10M,"вітер на 10 м, м/с",ознака,0.0,773,0.880,3.815,14.950
6,PRECTOTCORR,"опади, мм",ознака,0.0,656,0.000,0.020,116.570
7,GWETTOP,вологість верхнього шару ґрунту,ознака,0.0,80,0.150,0.600,0.960
8,VPD,дефіцит вологи повітря,ознака,0.0,3299,0.028,0.658,5.738
9,dry_days,сухих днів поспіль,ознака,0.0,44,0.000,3.000,43.000


## `h3_risk_cells`

In [5]:
o = report.dataset_eda('h3_risk_cells')
print(o['short'])
for s in o['seen']:
    print('·', s)
pd.DataFrame(o['columns'])[['n', 'doc', 'role', 'miss', 'uniq', 'min', 'med', 'max']]

13 344 гекси парку з шістьма різними лейблами. У 2022 згоріло 9 %, у 2023 22 %: тренування й тест дуже різні.
· Головна проблема видна вже тут: лейбл 2022 покриває 9 % парку, а 2023 22 %. Обидва рази згоріли лише 338 гекси з 1194.
· Три джерела про 2023 розходяться: згода FIRMS і Hansen 88 %, Hansen і NDVI 93 %.
· Лейбл 2024 порожній, у 2025 лише 327 гекс: після 2023 великих пожеж у парку не було, тому майбутніх перевірок ризику майже немає.
· Найсильніші ознаки для 2022: до межі парку, м (0,63), до точки FIRMS, м, станом на 2022 (0,63). Близькість до точок FIRMS це майже той самий вогонь, тому вона не доводить причину.
· Ознаки мають три копії, 2022, 2023 і 2026: кожна знає лише минуле свого року.


,n,doc,role,miss,uniq,min,med,max
0,park,,інше,0.0,3,NaN,NaN,NaN
1,h3,ідентифікатор гекса H3,ключ,0.0,13344,NaN,NaN,NaN
2,parent6,гекс-батько рівня 6,спліт,0.0,78,NaN,NaN,NaN
3,parent7,гекс-батько рівня 7,спліт,0.0,365,NaN,NaN,NaN
4,elev,"висота, м",ознака,0.0,12697,-3.764,10.208,564.565
5,slope,"ухил, °",ознака,0.0,12422,0.000,0.900,24.575
6,southness,наскільки схил південний,ознака,0.0,12145,-0.992,0.057,0.977
7,treecover,"лісистість 2000 року, %",ознака,0.0,6107,0.000,0.102,99.819
8,d_road,"до дороги, м",ознака,0.0,13344,0.795,557.661,4362.161
9,d_village,"до села, м",ознака,0.0,13343,0.795,1632.986,10921.199


## `logging_gr_cells`

In [6]:
o = report.dataset_eda('logging_gr_cells')
print(o['short'])
for s in o['seen']:
    print('·', s)
pd.DataFrame(o['columns'])[['n', 'doc', 'role', 'miss', 'uniq', 'min', 'med', 'max']]

106 752 рядків, але рубок лише 150, 0,1 %. У 2020 їх немає взагалі, у 2023 і 2024 по одній.
· Це 13344 гексів по 8 років: рядки не незалежні.
· Рік 2018 дає 45 % усіх рубок. Тест 2021–2024 має лише 35.
· Гексів із рубкою 145: повторів мало, тому «де рубали торік» працює слабко.
· Найсильніші ознаки до 2020: лісистість 2000 року, % (0,89), ухил, ° (0,79). Сила низька: рубки слабко пов'язані з ландшафтом.
· Втрата ≥ 0,5 га це поріг. Дрібніші втрати сюди не входять.


,n,doc,role,miss,uniq,min,med,max
0,park,,інше,0.0,3,NaN,NaN,NaN
1,h3,ідентифікатор гекса H3,ключ,0.0,13344,NaN,NaN,NaN
2,parent6,гекс-батько рівня 6,спліт,0.0,78,NaN,NaN,NaN
3,elev,"висота, м",ознака,0.0,12389,-3.764,10.208,564.565
4,slope,"ухил, °",ознака,0.0,10127,0.000,0.899,24.575
5,treecover,"лісистість 2000 року, %",ознака,0.0,6066,0.000,0.102,99.819
6,d_road,"до дороги, м",ознака,0.0,13340,0.795,557.661,4362.161
7,d_village,"до села, м",ознака,0.0,13341,0.795,1632.986,10921.199
8,track_km_km2,"густота лісових доріг, км/км²",ознака,0.0,1695,0.000,3.255,37.370
9,year,рік,час,0.0,8,2017.000,2020.500,2024.000


Решта датасетів: `report.dataset_eda('<назва>')` для будь-якої назви зі списку вище.